# 01 — Load Data (A1) + Clean & Merge (A2)
SIH26182 — Duo A (Data & ML)

**A1 goal:** Elliptic dataset downloaded and loaded, shapes confirmed.
**A2 goal:** One clean training-ready table, unknowns dropped, saved as `merged_data.csv`.


## A1.1 — Download the dataset
Uses `kagglehub` (no manual zip handling needed). If this fails in your environment, download `ellipticco/elliptic-data-set` manually from Kaggle and unzip it into `./elliptic_data/`.

In [15]:
# pip install kagglehub pandas networkx scikit-learn joblib matplotlib --quiet
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("ellipticco/elliptic-data-set")
print("Dataset downloaded to:", path)
print(os.listdir(path))


Dataset downloaded to: C:\Users\pabba\.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1
['elliptic_bitcoin_dataset']


**Fallback (manual download):** if `kagglehub` isn't available, set `path` to wherever you unzipped the Kaggle download, e.g. `path = "./elliptic_data"`, then re-run the cell below. Don't spend more than 15 minutes on this — it's step A1's known time sink.

## A1.2 — Load the three CSVs

In [16]:
import glob

def find_file(root, filename):
    matches = glob.glob(os.path.join(root, "**", filename), recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} anywhere under {root}")
    return matches[0]

features_path = find_file(path, "elliptic_txs_features.csv")
classes_path  = find_file(path, "elliptic_txs_classes.csv")
edges_path    = find_file(path, "elliptic_txs_edgelist.csv")

print(features_path)
print(classes_path)
print(edges_path)

# The features file has no header row in the original release; classes/edges do.
features_df = pd.read_csv(features_path, header=None)
classes_df  = pd.read_csv(classes_path)
edges_df    = pd.read_csv(edges_path)

feature_cols = ["txId", "time_step"] + [f"feat_{i}" for i in range(1, features_df.shape[1] - 1)]
features_df.columns = feature_cols

print("features_df shape:", features_df.shape)
print("classes_df shape:", classes_df.shape)
print("edges_df shape:", edges_df.shape)

C:\Users\pabba\.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1\elliptic_bitcoin_dataset\elliptic_txs_features.csv
C:\Users\pabba\.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1\elliptic_bitcoin_dataset\elliptic_txs_classes.csv
C:\Users\pabba\.cache\kagglehub\datasets\ellipticco\elliptic-data-set\versions\1\elliptic_bitcoin_dataset\elliptic_txs_edgelist.csv
features_df shape: (203769, 167)
classes_df shape: (203769, 2)
edges_df shape: (234355, 2)


In [17]:
print("Class distribution (raw):")
print(classes_df["class"].value_counts())
print()
print("features_df head:")
display(features_df.head())
print()
print("classes_df head:")
display(classes_df.head())
print()
print("edges_df head:")
display(edges_df.head())


Class distribution (raw):
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

features_df head:


,txId,time_step,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_156,feat_157,feat_158,feat_159,feat_160,feat_161,feat_162,feat_163,feat_164,feat_165
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162097,...,-0.562153,-0.600999,1.461330,1.461369,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162112,...,0.947382,0.673103,-0.979074,-0.978556,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.162749,...,0.670883,0.439728,-0.979074,-0.978556,-0.098889,-0.106715,-0.131155,-0.183671,-0.120613,-0.119792
3,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.577099,-0.613614,0.241128,0.241406,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792
4,230460314,1,1.011523,-0.081127,-1.201369,1.153668,0.333276,1.312656,-0.061584,-0.163523,...,-0.511871,-0.400422,0.517257,0.579382,0.018279,0.277775,0.326394,1.293750,0.178136,0.179117



classes_df head:


,txId,class
0,230425980,unknown
1,5530458,unknown
2,232022460,unknown
3,232438397,2
4,230460314,unknown



edges_df head:


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


✅ **Verify (A1):** ~203,769 rows in `features_df`/`classes_df` (one row per transaction), and the class column shows a clear imbalance (class `"2"`=licit dominates, class `"1"`=illicit is the minority, class `"unknown"`=unlabeled majority).

## A2 — Merge, drop unknowns, map labels

In [18]:
merged_df = features_df.merge(classes_df, left_on="txId", right_on="txId", how="inner")

# Drop unlabeled rows (class == "unknown")
merged_df = merged_df[merged_df["class"] != "unknown"].copy()

# Map to readable labels: class '1' -> illicit, class '2' -> licit
label_map = {"1": "illicit", "2": "licit"}
merged_df["label"] = merged_df["class"].astype(str).map(label_map)
merged_df = merged_df.drop(columns=["class"])

print("Final merged shape:", merged_df.shape)
print(merged_df["label"].value_counts())
print(merged_df["label"].value_counts(normalize=True))


Final merged shape: (46564, 168)
label
licit      42019
illicit     4545
Name: count, dtype: int64
label
licit      0.902392
illicit    0.097608
Name: proportion, dtype: float64


In [19]:
merged_df.to_csv("merged_data.csv", index=False)
print("Saved merged_data.csv with shape", merged_df.shape)


Saved merged_data.csv with shape (46564, 168)


✅ **Verify (A2):** Final labeled dataset ≈ 46,000 rows, illicit ≈ 10% of that (minority class).

⚠️ **If the merge produces 0 rows:** the join column name or dtype is wrong — print `features_df.columns` and `classes_df.columns` and confirm both `txId` columns are the same dtype (int vs str) before re-running.